# Teste de Transcrição com Whisper no Groq

Este notebook demonstra como usar a API do Groq para transcrever arquivos de áudio diretamente.

Embora o Whispher local tenha modelos como `tiny`, `small`, `base`, o Groq oferece modelos otimizados como `whisper-large-v3` e `whisper-large-v3-turbo` (que é mais rápido e eficiente, ideal para testes rápidos).

In [1]:
import os
import sys
from dotenv import load_dotenv
from groq import Groq

# Carregar variáveis de ambiente do arquivo .env (assumindo que está na raiz do projeto)
load_dotenv(os.path.join("..", ".env"))

# Verificar chave de API
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    print("⚠️ GROQ_API_KEY não encontrada. Verifique seu arquivo .env na raiz do projeto.")
else:
    print("✅ API Key encontrada.")

✅ API Key encontrada.


In [2]:
client = Groq(
    api_key=api_key,
)

# Função da transcrição

In [3]:
def transcrever_groq(audio_path, model="whisper-large-v3-turbo"):
    """
    Transcreve um arquivo de áudio usando a API do Groq.
    Args:
        audio_path (str): Caminho para o arquivo de áudio.
        model (str): Modelo a ser usado. Padrão: whisper-large-v3-turbo.
    """
    print(f"🚀 Iniciando transcrição de '{audio_path}' usando [{model}]...")
    
    if not os.path.exists(audio_path):
        return f"❌ Arquivo não encontrado: {audio_path}"
    
    try:
        # Abrir o arquivo em modo binário de leitura
        with open(audio_path, "rb") as file:
            # A API aceita o arquivo como uma tupla (nome, bytes)
            transcription = client.audio.transcriptions.create(
                file=(os.path.basename(audio_path), file.read()),
                model=model,
                response_format="text", # Retorna o texto diretamente
                language="pt"           # Definindo idioma para Português
            )
        return transcription
    except Exception as e:
        return f"❌ Erro na API: {e}"


# Teste com a receita

In [4]:
# Definir caminho do arquivo de teste
# Estamos na pasta Notebooks, então voltamos um nível para acessar 'data'
arquivo_teste = "../data/audio_receita.mp3"

# Verificar se o arquivo existe, caso contrário tenta listar o diretório para debug
if not os.path.exists(arquivo_teste):
    print(f"⚠️ Arquivo {arquivo_teste} não encontrado.")
    print("Arquivos em ../data:", os.listdir("../data") if os.path.exists("../data") else "Pasta data não encontrada")
else:
    # Executar transcrição
    texto = transcrever_groq(arquivo_teste, model="whisper-large-v3-turbo")

    print("\n--- Resultado da Transcrição ---\n")
    print(texto)

🚀 Iniciando transcrição de '../data/audio_receita.mp3' usando [whisper-large-v3-turbo]...

--- Resultado da Transcrição ---

 Olá, tudo bem? Meu nome é Jaine Figueira, eu sou chefe de cozinha e hoje eu vou fazer um doce italiano maravilhoso que se chama Zabayone. É uma coisa muito fácil de se fazer, tá? Aqui no Brasil A gente pode chamar de uma gemada, mas é uma gemada chique, uma gemada, né? Ela é elaborada. Porque a gente vai usar nela um vinho licoroso. Aqui hoje eu tô com um vinho Marsala, mas também você pode usar um vinho santo, pode usar um xerês, pode usar um vinho do Porto, né? Que é mais comum, mais fácil de achar. A gente tá usando o Marsala. A Marsala tá na moda, né? É a cor da moda. Não é nada mais do que um vinho que a gente chama de cor vinho. Bordeaux. Bordeaux também é um tipo de vinho. Marsala é um tipo de vinho. Então a cor é vinho, tá? Então a gente tem vários nomes pra cor. O vinho Marsala é um vinho bem adocicado, com um sabor bem peculiar dele. Então cada variaçã

# Teste com o Podcast

In [5]:
arquivo_teste = "../data/audio_podcast.mp3"

# Verificar se o arquivo existe, caso contrário tenta listar o diretório para debug
if not os.path.exists(arquivo_teste):
    print(f"⚠️ Arquivo {arquivo_teste} não encontrado.")
    print("Arquivos em ../data:", os.listdir("../data") if os.path.exists("../data") else "Pasta data não encontrada")
else:
    # Executar transcrição
    texto = transcrever_groq(arquivo_teste, model="whisper-large-v3-turbo")

    print("\n--- Resultado da Transcrição ---\n")
    print(texto)

🚀 Iniciando transcrição de '../data/audio_podcast.mp3' usando [whisper-large-v3-turbo]...

--- Resultado da Transcrição ---

❌ Erro na API: Error code: 413 - {'error': {'message': 'Request Entity Too Large', 'type': 'invalid_request_error', 'code': 'request_too_large'}}


# Necessário quebrar o audio quando for grande

In [6]:
from pydub import AudioSegment
import os

def dividir_audio(audio_path, chunk_min=5, output_dir="chunks_audio"):
    os.makedirs(output_dir, exist_ok=True)

    audio = AudioSegment.from_file(audio_path)
    duracao_ms = len(audio)
    chunk_ms = chunk_min * 60 * 1000

    caminhos = []

    for i in range(0, duracao_ms, chunk_ms):
        chunk = audio[i:i + chunk_ms]
        nome_chunk = f"chunk_{i//chunk_ms:03d}.mp3"
        caminho = os.path.join(output_dir, nome_chunk)
        chunk.export(caminho, format="mp3")
        caminhos.append(caminho)

    return caminhos



In [7]:
def transcrever_groq(audio_path, model="whisper-large-v3-turbo"):
    if not os.path.exists(audio_path):
        raise FileNotFoundError(audio_path)

    with open(audio_path, "rb") as file:
        transcription = client.audio.transcriptions.create(
            file=(os.path.basename(audio_path), file),
            model=model,
            response_format="text",
            language="pt"
        )

    return transcription


## Teste com receita

In [8]:
audio_original = "../data/audio_receita.mp3"

chunks = dividir_audio(audio_original, chunk_min=5)

In [9]:
chunks

['chunks_audio\\chunk_000.mp3',
 'chunks_audio\\chunk_001.mp3',
 'chunks_audio\\chunk_002.mp3']

In [10]:
transcricoes = []

for chunk in chunks:
    print(f"🎧 Transcrevendo {chunk}")
    texto = transcrever_groq(chunk)
    transcricoes.append(texto)

texto_final_2 = "\n".join(transcricoes)


🎧 Transcrevendo chunks_audio\chunk_000.mp3


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `whisper-large-v3-turbo` in organization `org_01kb1etjeqetstn38w64kmvkfj` service tier `on_demand` on seconds of audio per hour (ASPH): Limit 7200, Used 7074, Requested 300. Please try again in 1m27s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'seconds', 'code': 'rate_limit_exceeded'}}

In [ ]:
texto_final_2

' Olá, tudo bem? Meu nome é Jaime Figueira, eu sou chefe de cozinha e hoje eu vou fazer um doce italiano maravilhoso que se chama Zabayone. É uma coisa muito fácil de se fazer, tá? Aqui no Brasil A gente pode chamar de uma gemada, mas é uma gemada chique, uma gemada, né? Ela é elaborada. Porque a gente vai usar nela um vinho licoroso. Aqui hoje eu tô com um vinho Marsala, mas também você pode usar um vinho santo, pode usar um xerês, pode usar um vinho do Porto, né? Que é mais comum, mais fácil de achar. A gente tá usando Marsala. Marsala tá na moda, né? É a cor da moda. Não é nada mais do que um vinho que a gente chama de cor vinho. Bordeaux. Bordeaux também é um tipo de vinho. Marsala é um tipo de vinho. Então a cor é vinho, tá? Então a gente tem vários nomes pra cor. O vinho Marsala é um vinho bem adocicado, com um sabor bem peculiar dele Ent cada varia desses vinhos licorosos que eu falei pra voc vai ter um tom Como a gente t falando de uma receita italiana vamos usar o vinho da Mar

# Texto podcast

In [11]:
sys.path.append("..")

from src.utils_audio import dividir_audio_ffmpeg
# Use a nova função (note que ela usa ffmpeg, então é super rápida)
audio_original = "../data/audio_podcast.mp3"
chunks = dividir_audio_ffmpeg(audio_original, chunk_min=10, output_dir="../data/chunks_audio")

chunks

🔪 Dividindo áudio em chunks de 10 minutos usando FFMPEG...
✅ Sucesso! 25 chunks criados em '../data/chunks_audio'.


['../data/chunks_audio\\chunk_000.mp3',
 '../data/chunks_audio\\chunk_001.mp3',
 '../data/chunks_audio\\chunk_002.mp3',
 '../data/chunks_audio\\chunk_003.mp3',
 '../data/chunks_audio\\chunk_004.mp3',
 '../data/chunks_audio\\chunk_005.mp3',
 '../data/chunks_audio\\chunk_006.mp3',
 '../data/chunks_audio\\chunk_007.mp3',
 '../data/chunks_audio\\chunk_008.mp3',
 '../data/chunks_audio\\chunk_009.mp3',
 '../data/chunks_audio\\chunk_010.mp3',
 '../data/chunks_audio\\chunk_011.mp3',
 '../data/chunks_audio\\chunk_012.mp3',
 '../data/chunks_audio\\chunk_013.mp3',
 '../data/chunks_audio\\chunk_014.mp3',
 '../data/chunks_audio\\chunk_015.mp3',
 '../data/chunks_audio\\chunk_016.mp3',
 '../data/chunks_audio\\chunk_017.mp3',
 '../data/chunks_audio\\chunk_018.mp3',
 '../data/chunks_audio\\chunk_019.mp3',
 '../data/chunks_audio\\chunk_020.mp3',
 '../data/chunks_audio\\chunk_021.mp3',
 '../data/chunks_audio\\chunk_022.mp3',
 '../data/chunks_audio\\chunk_023.mp3',
 '../data/chunks_audio\\chunk_024.mp3']

In [15]:
import time
from src.utils_audio import transcrever_com_retry

# ... seu código de setup anterior ...

MODELO = "whisper-large-v3-turbo"
transcricoes = []
print(f"🚀 Iniciando transcrição de {len(chunks)} chunks com fallback automático...")
for i, chunk in enumerate(chunks):
    print(f"[{i+1}/{len(chunks)}] 🎧 Transcrevendo: {chunk}")
    
    try:
        # Usa a função importada do módulo recarregado
        texto = transcrever_com_retry(client, chunk, model=MODELO)
        
        if texto:
            transcricoes.append(texto)
            # Salvar backup parcial para segurança
            nome_txt = chunk.replace(".mp3", ".txt")
            with open(nome_txt, "w", encoding="utf-8") as f:
                f.write(texto)
        else:
            print(f"⚠️ AVISO: Transcrição vazia para {chunk}")
            
    except Exception as e:
        print(f"❌ Falha crítica no loop ao processar {chunk}.")
        print(f"   Erro original: {e}")
        print(f"   Tipo do erro: {type(e)}")
        continue
texto_final_podcast = "\n".join(transcricoes)
print("✅ Transcrição completa!")

🚀 Iniciando transcrição de 25 chunks com fallback automático...
[1/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_000.mp3
[2/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_001.mp3
[3/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_002.mp3
[4/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_003.mp3
[5/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_004.mp3
[6/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_005.mp3
[7/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_006.mp3
[8/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_007.mp3
[9/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_008.mp3
⚠️ Rate Limit atingido para 'chunk_008.mp3'.
⏳ Aguardando 285.0 segundos para liberar cota da API...
🔄 Retomando transcrição...
[10/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_009.mp3
⚠️ Rate Limit atingido para 'chunk_009.mp3'.
⏳ Aguardando 296.0 segundos para liberar cota da API...
🔄 Retomando transcrição...
[11/25] 🎧 Transcrevendo: ../data/chunks_audio\chunk_010.mp3
⚠️ Rate Limit atingido para '

In [16]:
texto_final_podcast

' E aí E aí E aí E aí E aí E a Vocês, Rodrigo Tchorro, na minha frente, meu brother, meu irmão, Roberto Andrade, filho Borracha, na mesa operando nosso diretor, Pedro Henrique. Você tá dormindo até agora, Robertinho? Rapaz, eu acabei de acordar, e você já acordou? Rapaz, que sacanagem você fez, hein? Vai lá no Instagram do Três Irmãos e vê o que o Robertinho arrumou comigo. Tava brincando com ele e pegou pesado, bicho. Vejam no Instagram do podcast Três Irmãos, vejam as TV sem excesso nas brincadeiras. Pra mim foi a mesma brincadeira, um vídeo de ir lá descansando, um vídeo de cá descansando. Só mostrou a forma que cada um descansa. Eu descanso, assim, sobre os braços. Você descansa um pouco mais espaçoso. Robertinho, mais um debate incrível no podcast Três Irmãos. Trabalhistas versus comunistas! Esse momento chegou... Tá de zoeira, tá de zoeira. Não vai ser um debate? Você vai proteger seu amiguinho aí? Não, Robertinho, não vou proteger não. Vai proteger seu amiguinho? Eu vou te falar